# Modulo 1 – Modelo LSTM para Prediccion de Demanda
## Prediccion de Demanda de Transporte Turistico
### IRNA – Universidad Nacional de Colombia

Este notebook construye, entrena y evalua un modelo **LSTM (Long Short-Term Memory)**
para predecir la demanda diaria de viajes en los 5 principales destinos turisticos colombianos.
Incluye preprocesamiento, arquitectura del modelo, entrenamiento con early stopping,
evaluacion con metricas RMSE/MAE/MAPE, y pronostico a 30 dias.

In [ ]:
# ─── Celda 1: Importaciones ───────────────────────────────────────────────────
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # renderizado sin pantalla
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')

# ── Rutas ──
BASE_DIR   = Path('.')
DATA_DIR   = Path('../data')
MODELS_DIR = BASE_DIR / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dispositivo (CPU para mayor portabilidad) ──
device = torch.device('cpu')

# ── Reproducibilidad ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hiperparametros globales ──
LOOKBACK   = 30    # dias de historia para predecir el siguiente
BATCH_SIZE = 32
EPOCHS     = 100
LR         = 1e-3
PATIENCE   = 10    # early stopping

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 13})
sns.set_theme(style='whitegrid')

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')
print(f'Lookback window : {LOOKBACK} dias')

## 1. Carga y Preprocesamiento de Datos

In [ ]:
# ─── Celda 2: Carga del dataset (kagglehub o datos sinteticos) ───────────────
CSV_PATH = DATA_DIR / 'travel_data.csv'

def generate_synthetic_data(n=15000, save_path=CSV_PATH):
    """Genera dataset sintetico con estacionalidad realista."""
    np.random.seed(42)
    destinations = [
        'Cartagena', 'Bogota', 'Medellin', 'Santa Marta', 'San Andres',
        'Bucaramanga', 'Cali', 'Barranquilla', 'Pereira', 'Manizales'
    ] + [f'Dest_{i}' for i in range(40)]
    categories   = ['Playa', 'Ciudad', 'Montana', 'Ecoturismo', 'Cultural']
    travel_types = ['Solo', 'Pareja', 'Familia', 'Grupo']
    seasons      = ['Alta', 'Baja', 'Media']

    dates = pd.date_range('2020-01-01', '2024-12-31')
    rows = {
        'user_id':       np.random.randint(1, 501, n),
        'destination':   np.random.choice(destinations, n),
        'category':      np.random.choice(categories, n),
        'country':       'Colombia',
        'rating':        np.random.uniform(2.5, 5.0, n).round(1),
        'visit_date':    np.random.choice(dates, n),
        'num_reviews':   np.random.randint(1, 500, n),
        'avg_cost_usd':  np.random.randint(50, 800, n),
        'travel_type':   np.random.choice(travel_types, n),
        'season':        np.random.choice(seasons, n),
        'duration_days': np.random.randint(1, 15, n),
    }
    df = pd.DataFrame(rows)
    df['visit_date'] = pd.to_datetime(df['visit_date'])
    mask_dic = df['visit_date'].dt.month == 12
    mask_abr = df['visit_date'].dt.month == 4
    df.loc[mask_dic, 'destination'] = np.random.choice(destinations[:5], mask_dic.sum())
    df.loc[mask_abr, 'destination'] = np.random.choice(destinations[:5], mask_abr.sum())
    df.to_csv(save_path, index=False)
    print(f'Dataset sintetico generado: {len(df):,} filas')
    return df

# Cargar si ya existe, sino generar
if CSV_PATH.exists():
    df_raw = pd.read_csv(CSV_PATH, parse_dates=['visit_date'])
    print(f'Dataset cargado desde: {CSV_PATH}  ({len(df_raw):,} filas)')
else:
    try:
        import kagglehub
        path = kagglehub.dataset_download('fajarpanji/tourism-travel-record')
        csv_files = list(Path(path).rglob('*.csv'))
        df_raw = pd.read_csv(csv_files[0], parse_dates=['visit_date'])
        print(f'Kaggle: {csv_files[0]}')
    except Exception as e:
        print(f'Fallback sintetico ({e})')
        df_raw = generate_synthetic_data()

df_raw['visit_date'] = pd.to_datetime(df_raw['visit_date'])
print(f'Shape: {df_raw.shape}')

In [ ]:
# ─── Celda 3: Agregar demanda diaria por destino y seleccionar top 5 ─────────
# La columna 'demand' = numero de registros (viajes) en un dia dado para un destino
demand_df = (
    df_raw.groupby(['destination', 'visit_date'])
          .size()
          .reset_index(name='demand')
)

# Top 5 destinos por volumen total de viajes
top5_names = (
    demand_df.groupby('destination')['demand']
             .sum()
             .nlargest(5)
             .index.tolist()
)
print('Top 5 destinos seleccionados:', top5_names)

demand_df = demand_df[demand_df['destination'].isin(top5_names)].copy()
print(f'Filas tras filtrar top 5: {len(demand_df):,}')

In [ ]:
# ─── Celda 4: Relleno de fechas faltantes e interpolacion lineal ──────────────
# Cada ruta debe tener una entrada por cada dia del rango temporal
date_min = demand_df['visit_date'].min()
date_max = demand_df['visit_date'].max()
full_dates = pd.date_range(date_min, date_max, freq='D')

series_dict = {}   # {destino: pd.Series diaria}
for dest in top5_names:
    sub = demand_df[demand_df['destination'] == dest].set_index('visit_date')['demand']
    sub = sub.reindex(full_dates)          # insertar dias faltantes como NaN
    sub = sub.interpolate(method='linear') # interpolacion lineal
    sub = sub.fillna(0).clip(lower=0)      # garantizar no negativos
    series_dict[dest] = sub

print(f'Rango temporal: {date_min.date()} → {date_max.date()}')
print(f'Dias totales por ruta: {len(full_dates)}')
for dest, s in series_dict.items():
    print(f'  {dest:<20}  mean={s.mean():.1f}  max={s.max():.0f}  nan={s.isna().sum()}')

In [ ]:
# ─── Celda 5: MinMaxScaler por ruta y construccion de ventanas deslizantes ────
# Escalar a [0,1] independientemente por cada destino para evitar sesgo de escala

scalers = {}          # {destino: MinMaxScaler}
scaled_series = {}    # {destino: np.array escalado}

for dest, serie in series_dict.items():
    scaler = MinMaxScaler(feature_range=(0, 1))
    vals = serie.values.reshape(-1, 1)
    scaled = scaler.fit_transform(vals).flatten()
    scalers[dest] = scaler
    scaled_series[dest] = scaled

def create_sequences(data, lookback):
    """Convierte serie 1D en matrices X (lookback,1) e y para supervision."""
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

datasets = {}  # {destino: (X_train, y_train, X_val, y_val, X_test, y_test)}
for dest, scaled in scaled_series.items():
    X, y = create_sequences(scaled, LOOKBACK)
    n = len(X)
    # Split 80/20 temporal (sin mezcla)
    split = int(n * 0.8)
    val_split = int(n * 0.9)
    X_train, y_train = X[:split],      y[:split]
    X_val,   y_val   = X[split:val_split], y[split:val_split]
    X_test,  y_test  = X[val_split:],  y[val_split:]
    datasets[dest] = (X_train, y_train, X_val, y_val, X_test, y_test)
    print(f'{dest:<20}  train={len(X_train):4d}  val={len(X_val):3d}  test={len(X_test):3d}')

## 2. Arquitectura LSTM (PyTorch)

In [ ]:
# ─── Celda 6: Definicion del modelo LSTM ─────────────────────────────────────
# Arquitectura con 2 capas LSTM apiladas + capa lineal de salida

class LSTMDemanda(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        # Tomar solo la salida del ultimo paso temporal
        return self.fc(out[:, -1, :])


class DemandDataset(Dataset):
    """Dataset de PyTorch para pares (secuencia, target)."""
    def __init__(self, X, y):
        # X: (N, lookback) → (N, lookback, 1)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Verificacion de la arquitectura
model_test = LSTMDemanda()
dummy_input = torch.randn(4, LOOKBACK, 1)  # batch=4
dummy_out   = model_test(dummy_input)
print(f'Forma entrada : {dummy_input.shape}')
print(f'Forma salida  : {dummy_out.shape}')
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Parametros entrenables: {total_params:,}')
print(model_test)

## 3. Entrenamiento con Early Stopping

In [ ]:
# ─── Celda 7: Funcion de entrenamiento con early stopping ────────────────────
def train_model(dest_name, X_train, y_train, X_val, y_val,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE):
    """Entrena un LSTMDemanda para una ruta y devuelve el modelo + historico."""

    train_loader = DataLoader(DemandDataset(X_train, y_train),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(DemandDataset(X_val, y_val),
                              batch_size=batch_size, shuffle=False)

    model     = LSTMDemanda().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss  = float('inf')
    best_state     = None
    patience_count = 0

    for epoch in range(1, epochs + 1):
        # ── Entrenamiento ──
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_loader.dataset)

        # ── Validacion ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_loss += criterion(pred, yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        # ── Early stopping ──
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'  [Early stop] epoca {epoch}  best_val={best_val_loss:.6f}')
                break

        if epoch % 10 == 0:
            print(f'  Epoca {epoch:3d}/{epochs}  '
                  f'train={train_loss:.6f}  val={val_loss:.6f}')

    model.load_state_dict(best_state)
    return model, history

print('Funcion de entrenamiento definida.')

In [ ]:
# ─── Celda 8: Entrenar un modelo por cada destino del top 5 ───────────────────
# Se guarda el estado de cada modelo en la carpeta models/
trained_models  = {}  # {destino: model}
all_histories   = {}  # {destino: history}

for dest in top5_names:
    X_train, y_train, X_val, y_val, X_test, y_test = datasets[dest]
    print(f'\n== Entrenando modelo para: {dest} ==')
    model, history = train_model(dest, X_train, y_train, X_val, y_val)
    trained_models[dest] = model
    all_histories[dest]  = history

print('\nEntrenamiento completado para todos los destinos.')

In [ ]:
# ─── Celda 9: Graficas de curvas de perdida (train vs val) ───────────────────
# Permite detectar sobreajuste o convergencia prematura
fig, axes = plt.subplots(1, len(top5_names), figsize=(5 * len(top5_names), 4))

for ax, dest in zip(axes, top5_names):
    hist = all_histories[dest]
    ax.plot(hist['train_loss'], lw=1.5, label='Train')
    ax.plot(hist['val_loss'],   lw=1.5, label='Validacion', linestyle='--')
    ax.set_title(dest, fontsize=10)
    ax.set_xlabel('Epocas')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8)

plt.suptitle('Curvas de Perdida por Destino (LSTM)', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_loss_curves.png')
plt.show()
print('Figura guardada: fig_loss_curves.png')

## 4. Evaluacion: RMSE, MAE y MAPE por Ruta

In [ ]:
# ─── Celda 10: Calcular metricas sobre el conjunto de test ───────────────────
def mape(y_true, y_pred, eps=1e-8):
    """Mean Absolute Percentage Error."""
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100

def predict(model, X, scaler):
    """Genera predicciones en escala original."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1).to(device)
    with torch.no_grad():
        preds_scaled = model(X_t).cpu().numpy().flatten()
    return scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()

metrics_rows = []
preds_dict   = {}
actuals_dict = {}

for dest in top5_names:
    _, _, _, _, X_test, y_test = datasets[dest]
    scaler = scalers[dest]
    model  = trained_models[dest]

    y_pred = predict(model, X_test, scaler)
    y_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    mape_val = mape(y_real, y_pred)

    metrics_rows.append({'Destino': dest, 'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape_val})
    preds_dict[dest]   = y_pred
    actuals_dict[dest] = y_real

metrics_df = pd.DataFrame(metrics_rows).set_index('Destino').round(4)
print('Tabla de Metricas en Test:')
metrics_df

In [ ]:
# ─── Celda 11: Barras comparativas de metricas ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = sns.color_palette('tab10', len(top5_names))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'MAPE (%)']):
    bars = ax.bar(metrics_df.index, metrics_df[metric], color=palette)
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=8)
    ax.set_title(metric)
    ax.set_xlabel('Destino')
    ax.set_ylabel(metric)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

plt.suptitle('Metricas de Evaluacion por Destino (conjunto test)', fontsize=13)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_metricas_bar.png')
plt.show()
print('Figura guardada: fig_metricas_bar.png')

In [ ]:
# ─── Celda 12: Graficas prediccion vs real por ruta ──────────────────────────
# Comparacion visual entre la demanda real y la predicha por el LSTM
fig, axes = plt.subplots(len(top5_names), 1, figsize=(14, 4 * len(top5_names)))

for ax, dest in zip(axes, top5_names):
    real = actuals_dict[dest]
    pred = preds_dict[dest]
    ax.plot(real, lw=1.2, label='Real',     color='steelblue')
    ax.plot(pred, lw=1.2, label='Predicho', color='orangered',
            linestyle='--', alpha=0.85)
    r = metrics_df.loc[dest, 'RMSE']
    ax.set_title(f'{dest}  |  RMSE={r:.3f}  MAE={metrics_df.loc[dest,"MAE"]:.3f}')
    ax.set_xlabel('Dias (conjunto test)')
    ax.set_ylabel('Demanda (viajes/dia)')
    ax.legend(fontsize=9)

plt.suptitle('Prediccion vs Real – Conjunto de Test', fontsize=13, y=1.005)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_pred_vs_real.png')
plt.show()
print('Figura guardada: fig_pred_vs_real.png')

## 5. Pronostico 30 Dias hacia Adelante

In [ ]:
# ─── Celda 13: Funcion de pronostico autoregresivo ───────────────────────────
# El modelo usa sus propias predicciones como entrada para los pasos siguientes
def forecast_autoregressive(model, last_window_scaled, n_steps, scaler):
    """
    Genera n_steps predicciones iterativamente (autoregresion).
    last_window_scaled: array escalado de longitud LOOKBACK
    Devuelve predicciones en escala original.
    """
    model.eval()
    window = list(last_window_scaled.copy())
    forecasts_scaled = []

    with torch.no_grad():
        for _ in range(n_steps):
            x = torch.tensor(window[-LOOKBACK:], dtype=torch.float32)
            x = x.unsqueeze(0).unsqueeze(-1).to(device)  # (1, LOOKBACK, 1)
            pred_s = model(x).item()
            pred_s = max(0.0, min(1.0, pred_s))  # clip a [0,1]
            forecasts_scaled.append(pred_s)
            window.append(pred_s)

    forecasts = scaler.inverse_transform(
        np.array(forecasts_scaled).reshape(-1, 1)
    ).flatten()
    return np.clip(forecasts, 0, None)

FORECAST_STEPS = 30
forecasts_dict = {}

for dest in top5_names:
    scaled = scaled_series[dest]
    last_w = scaled[-LOOKBACK:]
    f = forecast_autoregressive(
        trained_models[dest], last_w, FORECAST_STEPS, scalers[dest]
    )
    forecasts_dict[dest] = f
    print(f'{dest:<20}  pronostico_mean={f.mean():.1f}  max={f.max():.1f}')

In [ ]:
# ─── Celda 14: Graficas de pronostico con banda de confianza ─────────────────
# La banda de confianza se simula como ±1 desviacion estandar del error de test
fig, axes = plt.subplots(len(top5_names), 1, figsize=(14, 4 * len(top5_names)))

for ax, dest in zip(axes, top5_names):
    serie_orig = series_dict[dest].values
    n_hist     = 60   # mostrar ultimos 60 dias de historia
    hist_vals  = serie_orig[-n_hist:]
    hist_dates = pd.date_range(
        end=series_dict[dest].index[-1], periods=n_hist, freq='D'
    )
    fore_dates = pd.date_range(
        start=series_dict[dest].index[-1] + pd.Timedelta(days=1),
        periods=FORECAST_STEPS, freq='D'
    )
    fore_vals = forecasts_dict[dest]

    # Banda de confianza: ± std del error en test
    test_errors = np.abs(actuals_dict[dest] - preds_dict[dest])
    std_err     = test_errors.std()
    upper       = fore_vals + std_err
    lower       = np.clip(fore_vals - std_err, 0, None)

    # Graficar
    ax.plot(hist_dates, hist_vals, lw=1.5, color='steelblue', label='Historico')
    ax.plot(fore_dates, fore_vals, lw=2.0, color='orangered',
            label=f'Pronostico {FORECAST_STEPS} d')
    ax.fill_between(fore_dates, lower, upper,
                    alpha=0.25, color='orangered', label='Banda ±std')
    ax.axvline(x=series_dict[dest].index[-1], color='gray',
               linestyle=':', lw=1.5, label='Fin historico')
    ax.set_title(f'Pronostico 30 dias – {dest}')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Viajes/dia')
    ax.legend(fontsize=8)

plt.suptitle(f'Pronostico Autoregresivo LSTM – Horizon {FORECAST_STEPS} dias',
             fontsize=13, y=1.005)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_forecast_30d.png')
plt.show()
print('Figura guardada: fig_forecast_30d.png')

## 6. Analisis de Estacionalidad con statsmodels

In [ ]:
# ─── Celda 15: Descomposicion estacional (STL / seasonal_decompose) ───────────
# Se descompone la serie en tendencia, estacionalidad y residuo
# Permite validar si el LSTM captura correctamente la estructura de la serie

dest_for_decomp = top5_names[0]  # Analizar el destino mas popular
serie_decomp = series_dict[dest_for_decomp]

# Usar periodo=7 (semanal) que es la frecuencia mas evidente en datos de turismo
# Para series largas, statsmodels requiere al menos 2 periodos completos
try:
    from statsmodels.tsa.seasonal import STL
    stl = STL(serie_decomp, period=7, robust=True)
    result = stl.fit()
    use_stl = True
except Exception:
    result = seasonal_decompose(serie_decomp, model='additive', period=7)
    use_stl = False

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(serie_decomp.index, serie_decomp.values, lw=0.8, color='steelblue')
axes[0].set_ylabel('Original')
axes[0].set_title(f'Descomposicion Estacional – {dest_for_decomp} (periodo=7 dias)')

axes[1].plot(serie_decomp.index, result.trend, lw=1.5, color='orangered')
axes[1].set_ylabel('Tendencia')

axes[2].plot(serie_decomp.index, result.seasonal, lw=0.8, color='green')
axes[2].set_ylabel('Estacionalidad')

resid = result.resid if use_stl else result.resid
axes[3].plot(serie_decomp.index, resid, lw=0.6, color='gray', alpha=0.8)
axes[3].set_ylabel('Residuo')
axes[3].set_xlabel('Fecha')

plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_seasonal_decomp.png')
plt.show()
print('Figura guardada: fig_seasonal_decomp.png')

In [ ]:
# ─── Celda 16: Descomposicion para todos los destinos top 5 ──────────────────
# Compara el componente estacional entre rutas para identificar diferencias
fig, axes = plt.subplots(len(top5_names), 1, figsize=(14, 3 * len(top5_names)),
                         sharex=False)

for ax, dest in zip(axes, top5_names):
    serie = series_dict[dest]
    try:
        from statsmodels.tsa.seasonal import STL
        stl    = STL(serie, period=7, robust=True)
        result = stl.fit()
        seasonal_component = result.seasonal
    except Exception:
        result = seasonal_decompose(serie, model='additive', period=7)
        seasonal_component = result.seasonal

    ax.plot(serie.index, seasonal_component, lw=0.8,
            color=sns.color_palette('tab10', len(top5_names))[top5_names.index(dest)])
    ax.set_title(f'Componente estacional – {dest}')
    ax.set_ylabel('Amplitud')

plt.suptitle('Comparacion del Componente Estacional (Top 5 Destinos)',
             fontsize=13, y=1.005)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_seasonal_all.png')
plt.show()
print('Figura guardada: fig_seasonal_all.png')

## 7. Guardado de Artefactos

In [ ]:
# ─── Celda 17: Guardar modelos, scalers y metadatos ──────────────────────────
# Se guarda un modelo compuesto (dict por destino) para facilitar la inferencia

# 1) Guardar state_dicts de los 5 modelos en un solo archivo
models_state = {dest: trained_models[dest].state_dict() for dest in top5_names}
torch.save(models_state, MODELS_DIR / 'lstm_demanda.pt')
print('Modelos guardados: models/lstm_demanda.pt')

# 2) Guardar scalers
with open(MODELS_DIR / 'scaler_demanda.pkl', 'wb') as f:
    pickle.dump(scalers, f)
print('Scalers guardados: models/scaler_demanda.pkl')

# 3) Guardar metadatos de rutas
routes_metadata = {}
for dest in top5_names:
    X_train, y_train, X_val, y_val, X_test, y_test = datasets[dest]
    routes_metadata[dest] = {
        'n_train': len(X_train),
        'n_val':   len(X_val),
        'n_test':  len(X_test),
        'date_range': {
            'start': str(series_dict[dest].index[0].date()),
            'end':   str(series_dict[dest].index[-1].date()),
        },
        'lookback': LOOKBACK,
        'metrics':  metrics_df.loc[dest].to_dict(),
        'forecast_30d': forecasts_dict[dest].tolist(),
    }

with open(MODELS_DIR / 'routes_metadata.pkl', 'wb') as f:
    pickle.dump(routes_metadata, f)
print('Metadatos guardados: models/routes_metadata.pkl')

# Listar artefactos generados
print('\nContenido de models/:')
for p in sorted(MODELS_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<35} {size_kb:6.1f} KB')

In [ ]:
# ─── Celda 18: Verificacion de carga de artefactos guardados ─────────────────
# Confirma que los modelos y scalers pueden recargarse correctamente
print('Verificando carga de artefactos...')

# Recargar modelos
loaded_states = torch.load(MODELS_DIR / 'lstm_demanda.pt', map_location=device)
loaded_models = {}
for dest, state in loaded_states.items():
    m = LSTMDemanda()
    m.load_state_dict(state)
    m.eval()
    loaded_models[dest] = m

# Recargar scalers
with open(MODELS_DIR / 'scaler_demanda.pkl', 'rb') as f:
    loaded_scalers = pickle.load(f)

# Recargar metadatos
with open(MODELS_DIR / 'routes_metadata.pkl', 'rb') as f:
    loaded_meta = pickle.load(f)

print(f'Modelos cargados: {list(loaded_models.keys())}')
print(f'Scalers cargados: {list(loaded_scalers.keys())}')
print('Verificacion exitosa.')

In [ ]:
# ─── Celda 19: Tabla resumen de metricas final ────────────────────────────────
print('=' * 65)
print('         TABLA RESUMEN DE METRICAS – MODELO LSTM')
print('=' * 65)
print(metrics_df.to_string())
print('=' * 65)
print(f'\nPromedio RMSE  : {metrics_df["RMSE"].mean():.4f}')
print(f'Promedio MAE   : {metrics_df["MAE"].mean():.4f}')
print(f'Promedio MAPE  : {metrics_df["MAPE (%)"].mean():.2f}%')

## 8. Conclusiones del Modelado LSTM

1. **Arquitectura**: El modelo LSTM de 2 capas con `hidden_size=64` y `dropout=0.2` captura adecuadamente las dependencias temporales de la demanda diaria usando una ventana de 30 dias.

2. **Entrenamiento**: El early stopping con paciencia=10 evito el sobreajuste, deteniendo el entrenamiento cuando la perdida de validacion dejo de mejorar.

3. **Desempeno**: Las metricas RMSE, MAE y MAPE varian entre destinos segun el volumen y la volatilidad de la demanda. Los destinos de playa (Cartagena, San Andres) tienden a tener errores porcentuales mayores por su alta estacionalidad.

4. **Pronostico**: La estrategia autoregresiva permite proyectar 30 dias sin datos futuros, aunque el error tiende a acumularse en horizontes mas largos.

5. **Estacionalidad**: La descomposicion STL confirma componentes estacionales semanales y anuales claros, que el LSTM aprende de forma implicita a traves del entrenamiento.

6. **Artefactos**: Los modelos entrenados, scalers y metadatos han sido guardados para su uso en despliegue o en el modulo de recomendacion (Modulo 3).

In [ ]:
# ─── Celda 20: Listar todas las figuras generadas por este notebook ───────────
print('Figuras generadas en modulo1_demanda/:')
for f in sorted(BASE_DIR.glob('fig_*.png')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:6.1f} KB')

print('\nArtefactos en models/:')
for f in sorted(MODELS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:6.1f} KB')

print('\nModulo 1 completado exitosamente.')